# 01 - Extraction and Cleaning

**Project:** AI-Powered Chatbot for Data Engineering Students (FYP, UiTM)
**Phase 6, Step 1:** Knowledge Base Preparation - extract text from the raw sources and clean it.

## What this notebook does
1. Reads the rules for each of the 7 sources (S01-S07) from `config/settings.py`. Nothing about paths, page ranges or cleaning rules is hardcoded here.
2. Extracts the text from the raw files in `data/raw/` (**read-only**: the raw files are never modified).
3. Applies the cleaning rules (strip front matter, header lines, bullet characters, wiki links, ...).
4. Prints a **BEFORE** (raw) and **AFTER** (cleaned) sample for every source.
5. Writes the cleaned text to `data/cleaned/` and a manifest file, `extraction_manifest.json`.

## What this notebook does NOT do
- No chunking (that is `02_chunking.ipynb`).
- **S03 extractor is NOT decided yet.** Section 5 compares PyMuPDF and pdfplumber side by side; both candidate outputs are written and the decision is made after review.

## Output format
- **PDF sources** become one `.txt` file per source. Each page starts with a marker line `=== page N ===` so that the page number can be kept as metadata during chunking.
- **Markdown / code sources (S05, S06)** keep their original folder structure under `data/cleaned/S05/` and `data/cleaned/S06/`, one file per original file.

## 1. Setup

In [1]:
import sys
import re
import json
import math
import shutil
import zipfile
import logging
import textwrap
from itertools import zip_longest
from pathlib import Path

import pandas as pd
import pdfplumber
import pymupdf

# The notebook lives in notebooks/, so walk up until the folder that holds config/ is found.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import settings  # all paths and rules come from here

# pdfplumber prints harmless "FontBBox" warnings for some PDFs; hide them.
logging.getLogger("pdfminer").setLevel(logging.ERROR)

# Display-only settings for this notebook (not part of the pipeline rules).
SAMPLE_CHARS = 400     # how many characters to show in BEFORE / AFTER samples
LONG_TOKEN_LEN = 20    # a "word" longer than this is suspicious (merged words) in the S03 comparison

settings.CLEANED_DIR.mkdir(parents=True, exist_ok=True)
summary_rows = []      # one row per output, printed as a table at the end
manifest = []          # one entry per output, saved to extraction_manifest.json

print("Project root :", settings.PROJECT_ROOT)
print("Raw data     :", settings.RAW_DIR)
print("Cleaned data :", settings.CLEANED_DIR)
print("Sources      :", ", ".join(settings.SOURCES))

Project root : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot
Raw data     : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot\data\raw
Cleaned data : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot\data\cleaned
Sources      : S01, S02, S03, S04, S05, S06, S07


## 2. Helper functions
Small, single-purpose functions used by every section below.

In [2]:
def page_numbers(total_pages, page_ranges):
    """Return the 1-based page numbers to process.

    page_ranges is a list of (first, last) tuples from settings.SOURCES, or None for all pages.
    """
    if page_ranges is None:
        return list(range(1, total_pages + 1))
    wanted = []
    for first, last in page_ranges:
        wanted.extend(range(first, min(last, total_pages) + 1))
    return wanted


def extract_pdf_pages(pdf_path, extractor, page_ranges):
    """Extract raw text page by page. Returns a list of (page_number, raw_text).

    extractor is "pdfplumber" or "pymupdf". The PDF is opened read-only.
    """
    pages = []
    if extractor == "pdfplumber":
        with pdfplumber.open(pdf_path) as pdf:
            for number in page_numbers(len(pdf.pages), page_ranges):
                pages.append((number, pdf.pages[number - 1].extract_text() or ""))
    elif extractor == "pymupdf":
        with pymupdf.open(pdf_path) as doc:
            for number in page_numbers(len(doc), page_ranges):
                pages.append((number, doc[number - 1].get_text("text") or ""))
    else:
        raise ValueError(f"Unknown extractor: {extractor}")
    return pages


def apply_source_rules(text, source):
    """Apply the source-specific rules from settings: strip_line_patterns and strip_characters."""
    for pattern in source.get("strip_line_patterns", []):
        regex = re.compile(pattern)
        text = "\n".join(line for line in text.splitlines() if not regex.match(line))
    for character in source.get("strip_characters", []):
        text = text.replace(character, "")
    return text


def normalize_pdf_text(text):
    """Tidy whitespace in PDF text: collapse repeated spaces, trim lines, limit blank lines."""
    text = re.sub(r"[ \t]+", " ", text)
    text = "\n".join(line.strip() for line in text.splitlines())
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_pdf_pages(raw_pages, source):
    """Clean every extracted page. Pages shorter than MIN_PAGE_CHARACTERS are dropped as empty.

    Returns (all_records, kept_records); each record keeps raw / rule-cleaned / final text.
    """
    records = []
    for number, raw in raw_pages:
        ruled = apply_source_rules(raw, source)
        records.append({"page": number, "raw": raw, "ruled": ruled, "clean": normalize_pdf_text(ruled)})
    kept = [r for r in records if len(r["clean"]) >= settings.MIN_PAGE_CHARACTERS]
    return records, kept


def write_pdf_text(source_id, kept_records, suffix=""):
    """Write cleaned PDF pages to data/cleaned/<id><suffix>.txt with '=== page N ===' markers."""
    out_path = settings.CLEANED_DIR / f"{source_id}{suffix}.txt"
    body = "\n\n".join(f"=== page {r['page']} ===\n{r['clean']}" for r in kept_records)
    out_path.write_text(body, encoding="utf-8")
    return out_path


def first_difference(a, b):
    """Index of the first character where a and b differ (None if identical)."""
    if a == b:
        return None
    return next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))


def pick_sample(records):
    """Pick the page that best shows the cleaning rules at work.

    Prefers a page where a rule changed text within the first SAMPLE_CHARS characters;
    otherwise the first page with enough text.
    """
    candidates = []
    for r in records:
        diff = first_difference(r["raw"], r["ruled"])
        if diff is not None and diff < SAMPLE_CHARS:
            candidates.append((diff, r))
    if candidates:
        return min(candidates, key=lambda c: c[0])[1], True
    for r in records:
        if len(r["clean"]) >= SAMPLE_CHARS:
            return r, False
    return records[0], False


def show_before_after(label, before, after, note=""):
    """Print the first SAMPLE_CHARS characters of the raw text and of the cleaned text."""
    print("=" * 90)
    print(label, f"({note})" if note else "")
    print("-" * 90)
    print("BEFORE (raw extracted text):")
    print(before[:SAMPLE_CHARS])
    print("-" * 90)
    print("AFTER (cleaned text):")
    print(after[:SAMPLE_CHARS])
    print("=" * 90)


def add_summary(source_id, fmt, processed, characters, outputs, note=""):
    """Record one row for the final summary table and one entry for the manifest."""
    summary_rows.append({
        "source": source_id,
        "format": fmt,
        "pages/files processed": processed,
        "characters extracted": characters,
        "output file(s) written": outputs,
    })

## 3. PDF textbooks: S01, S02, S04 and S07
These four sources use the same procedure: extract with the extractor named in `settings.py`, keep only the configured page ranges, clean, and write one `.txt` file.

Only S07 has a line-pattern rule (the browser header with a date and time); for the others the AFTER sample differs from BEFORE only by whitespace tidying.

In [3]:
def process_pdf_source(source_id):
    """Run the full extract -> clean -> write procedure for one PDF source and show a sample."""
    source = settings.SOURCES[source_id]
    pdf_path = settings.RAW_DIR / source["path"]

    raw_pages = extract_pdf_pages(pdf_path, source["extractor"], source.get("page_ranges"))
    records, kept = clean_pdf_pages(raw_pages, source)
    out_path = write_pdf_text(source_id, kept)

    sample, rule_applied = pick_sample(records)
    note = f"page {sample['page']}, " + ("a cleaning rule changed this page" if rule_applied else "whitespace tidying only")
    show_before_after(f"{source_id} - {source['name']}", sample["raw"], sample["clean"], note)

    chars = sum(len(r["clean"]) for r in kept)
    print(f"{source_id}: extractor={source['extractor']}, pages in range={len(records)}, "
          f"empty pages dropped={len(records) - len(kept)}, characters={chars:,}")
    print(f"Written: {out_path.relative_to(settings.PROJECT_ROOT)}\n")

    add_summary(source_id, "PDF", f"{len(kept)} pages ({len(records)} in range)", chars,
                str(out_path.relative_to(settings.PROJECT_ROOT)))
    manifest.append({"source_id": source_id, "source_type": source["source_type"], "format": "pdf",
                     "extractor": source["extractor"], "output": out_path.name,
                     "pages": len(kept), "characters": chars, "page_markers": True})
    return records, kept

In [4]:
s01_records, s01_kept = process_pdf_source("S01")

S01 - BCcampus - Database Design (2nd edition, Watt) (page 8, whitespace tidying only)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
About the Book
Database Design – 2nd Edition is a remix and adaptation based on Adrienne Watt’s book Database Design. Works that are
part of the remix for this book are listed at the end of each chapter. For information about what was changed in this
adaptation, refer to the copyright statement in the copyright section or on the webbook homepage.
BCcampus Open Education began in 2012 as the B.C. O
------------------------------------------------------------------------------------------
AFTER (cleaned text):
About the Book
Database Design – 2nd Edition is a remix and adaptation based on Adrienne Watt’s book Database Design. Works that are
part of the remix for this book are listed at the end of each chapter. For information about what was changed in this
adaptation, refer to the cop

In [5]:
s02_records, s02_kept = process_pdf_source("S02")

S02 - OpenStax - Principles of Data Science (page 19, whitespace tidying only)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
1
What Are Data and Data Science?
Figure 1.1 Petroglyphs are one of the earliest types of data generated by humanity, providing vital information about the daily life of
the people who created them. (credit: modification of work “Indian petroglyphs (~100 B.C. to ~1540 A.D.) (Newspaper Rock,
southeastern Utah, USA) 24” by James St. John/Flickr, CC BY 2.0)
Chapter Outline
1.1 What Is Data Science?
1
------------------------------------------------------------------------------------------
AFTER (cleaned text):
1
What Are Data and Data Science?
Figure 1.1 Petroglyphs are one of the earliest types of data generated by humanity, providing vital information about the daily life of
the people who created them. (credit: modification of work “Indian petroglyphs (~100 B.C. to ~1540 A.D.) (Newspaper R

In [6]:
s04_records, s04_kept = process_pdf_source("S04")

S04 - OpenStax - Introduction to Computer Science (page 369, whitespace tidying only)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
8
Data Management
Figure 8.1 Big data taxonomy includes data providers, data consumers, data owners, and data viewers while providing the data flow
between them. (credit: modification of “DARPA Big Data” by Defense Advanced Research Projects Agency (DARPA)/Wikimedia
Commons, Public Domain)
Chapter Outline
8.1 Data Management Focus
8.2 Data Management Systems
8.3 Relational Database Management Syst
------------------------------------------------------------------------------------------
AFTER (cleaned text):
8
Data Management
Figure 8.1 Big data taxonomy includes data providers, data consumers, data owners, and data viewers while providing the data flow
between them. (credit: modification of “DARPA Big Data” by Defense Advanced Research Projects Agency (DARPA)/Wikimedia
Commons, Publ

### S07 - header stripping
S07 is a browser printout, so each page starts with a line like `10/6/26, 2:20 PM 1.4 Data Quality ...`. The rule `strip_line_patterns` in `settings.py` removes it. Check that the BEFORE sample shows the header and the AFTER sample does not.

In [7]:
s07_records, s07_kept = process_pdf_source("S07")
remaining = [line for r in s07_kept for line in r["clean"].splitlines()
             if re.match(settings.SOURCES["S07"]["strip_line_patterns"][0], line)]
print("Header-like lines still present in the cleaned S07 text:", len(remaining))

S07 - Luna Reyes et al. - Data Analytics for Public Policy, section 1.4 (page 1, a cleaning rule changed this page)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
10/6/26, 2:20 PM 1.4 Data Quality and Data Types – Data Analytics for Public Policy and Management
Home Read Sign
Search in book …
in
Want to create or adapt books like this? Learn more⧉ about how Pressbooks
supports open publishing practices.
DATA ANALYTICS FOR PUBLIC POLICY AND MANAGEMENT
CONTENTS
1.4 Data Quality and Data Types
Understanding data and data quality are basic requirements to work 
------------------------------------------------------------------------------------------
AFTER (cleaned text):
Home Read Sign
Search in book …
in
Want to create or adapt books like this? Learn more⧉ about how Pressbooks
supports open publishing practices.
DATA ANALYTICS FOR PUBLIC POLICY AND MANAGEMENT
CONTENTS
1.4 Data Quality and Data Types
Understanding da

## 4. S05 - Data Engineering Wiki (Markdown in a ZIP)
Rules from `settings.py`:
- read only the `.md` files, skip `README.md` and `LICENSE`;
- remove the front matter block (`--- ... ---`) at the top of each page;
- resolve `[[wiki links]]` to plain text (`[[Target|Alias]]` becomes `Alias`) and remove `![[image embeds]]`.

The ZIP is read in memory; nothing is extracted into `data/raw/`.

In [8]:
FRONTMATTER_RE = re.compile(r"\A---\s*\n.*?\n---\s*(?:\n|\Z)", re.DOTALL)
EMBED_RE = re.compile(r"!\[\[[^\]]*\]\]")        # ![[image.png|100]]
WIKILINK_RE = re.compile(r"\[\[([^\]]*)\]\]")     # [[Target]] or [[Target|Alias]]


def list_zip_files(zip_path):
    """Return [(ZipInfo, relative_path)] for every file, dropping the ZIP's single top-level folder."""
    archive = zipfile.ZipFile(zip_path)
    entries = []
    for info in archive.infolist():
        if info.is_dir():
            continue
        parts = info.filename.split("/")
        if len(parts) > 1:
            entries.append((info, "/".join(parts[1:])))
    return archive, entries


def wiki_link_to_text(match):
    """Turn the inside of a [[wiki link]] into readable text (alias if present, else the target name)."""
    target, _, alias = match.group(1).partition("|")
    alias = alias.strip()
    if alias:
        return alias
    target = target.rstrip("\\").split("#")[0]     # drop '#Heading' part and any escape character
    return target.split("/")[-1].strip()            # drop folder prefix such as 'Concepts/'


def clean_markdown(text, source):
    """Apply the Markdown cleaning rules chosen in settings (front matter, wiki links) and tidy whitespace."""
    text = text.replace("\r\n", "\n")
    if source.get("strip_frontmatter"):
        text = FRONTMATTER_RE.sub("", text, count=1)
    if source.get("resolve_wiki_links"):
        text = EMBED_RE.sub("", text)
        text = WIKILINK_RE.sub(wiki_link_to_text, text)
    text = "\n".join(line.rstrip() for line in text.splitlines())
    return re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"


def reset_output_folder(folder):
    """Delete a previously generated output folder (only ever inside data/cleaned/) and recreate it."""
    assert settings.CLEANED_DIR.resolve() in folder.resolve().parents, "refusing to touch a folder outside data/cleaned"
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True)

In [9]:
s05 = settings.SOURCES["S05"]
archive, entries = list_zip_files(settings.RAW_DIR / s05["path"])

selected = [(info, rel) for info, rel in entries
            if Path(rel).suffix.lower() in s05["include_extensions"]
            and Path(rel).name not in s05["exclude_filenames"]]
print(f"S05: {len(entries)} files in the ZIP, {len(selected)} selected")

out_dir = settings.CLEANED_DIR / "S05"
reset_output_folder(out_dir)

s05_chars = 0
sample_shown = False
for info, rel in selected:
    raw = archive.read(info).decode("utf-8", errors="replace")
    cleaned = clean_markdown(raw, s05)
    target = out_dir / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(cleaned, encoding="utf-8")
    s05_chars += len(cleaned)
    manifest.append({"source_id": "S05", "source_type": s05["source_type"], "format": "markdown",
                     "original_path": rel, "output": f"S05/{rel}", "characters": len(cleaned)})
    # Show the first file that actually contains front matter AND wiki links
    if not sample_shown and raw.startswith("---") and "[[" in raw:
        show_before_after(f"S05 - {rel}", raw, cleaned, "front matter removed, wiki links resolved")
        sample_shown = True

print(f"S05: wrote {len(selected)} files, {s05_chars:,} characters")
add_summary("S05", "Markdown (ZIP)", f"{len(selected)} files", s05_chars, f"{len(selected)} files in data/cleaned/S05/")

leftover = sum(1 for p in out_dir.rglob("*.md") if "[[" in p.read_text(encoding="utf-8") or p.read_text(encoding="utf-8").startswith("---"))
print("Cleaned S05 files that still contain '[[' or start with front matter:", leftover)

S05: 79 files in the ZIP, 76 selected
S05 - Guides/Testing Your Data Pipeline.md (front matter removed, wiki links resolved)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
---
Aliases: []
Tags: [evergreen]
publish: true
---

## Why Is Testing Important?

Writing tests have a few key benefits for data engineers. They can help you sleep better at night knowing you're less likely to get called into an emergency for a data issue. They can be used to help yourself and stakeholders get on the same page and understand the data you're working with. And one of the most impor
------------------------------------------------------------------------------------------
AFTER (cleaned text):
## Why Is Testing Important?

Writing tests have a few key benefits for data engineers. They can help you sleep better at night knowing you're less likely to get called into an emergency for a data issue. They can be used to help yourself an

## 5. S03 - Helsinki lecture slides: extractor comparison (decision pending)
S03 is the problem source: with pdfplumber the words of some slides run together. The two candidate extractors are run on the **same pages** and compared:
1. side by side on sample pages, and
2. with a simple measure: how many "words" are suspiciously long (merged words), per 1000 words.

**No extractor is chosen in this notebook.** Both cleaned versions are written (`S03_pymupdf.txt` and `S03_pdfplumber.txt`) so that you can decide after review; the unused file is then deleted.

In [10]:
s03 = settings.SOURCES["S03"]
s03_candidates = [s03["extractor"], s03["fallback_extractor"]]     # taken from settings.py
print("S03 candidates:", s03_candidates)

s03_results = {}
for name in s03_candidates:
    raw_pages = extract_pdf_pages(settings.RAW_DIR / s03["path"], name, s03.get("page_ranges"))
    records, kept = clean_pdf_pages(raw_pages, s03)       # same cleaning rules for both (strips the bullet characters)
    s03_results[name] = {"records": records, "kept": kept}
    print(f"{name:11s}: {len(records)} pages, {len(kept)} kept, {sum(len(r['clean']) for r in kept):,} characters")

S03 candidates: ['pymupdf', 'pdfplumber']


pymupdf    : 138 pages, 138 kept, 58,435 characters


pdfplumber : 138 pages, 138 kept, 57,206 characters


In [11]:
def suspicious_tokens(text):
    """Words longer than LONG_TOKEN_LEN characters that are not URLs, paths or numbers (likely merged words)."""
    tokens = re.findall(r"\S+", text)
    bad = [t for t in tokens if len(t) > LONG_TOKEN_LEN and not re.search(r"[\d/@:_\\\-]|https?|www", t)]
    return tokens, bad


rows = []
for name, res in s03_results.items():
    all_text = "\n".join(r["clean"] for r in res["kept"])
    tokens, bad = suspicious_tokens(all_text)
    rows.append({"extractor": name, "words": len(tokens), "suspicious long words": len(bad),
                 "per 1000 words": round(1000 * len(bad) / max(len(tokens), 1), 1),
                 "avg word length": round(sum(map(len, tokens)) / max(len(tokens), 1), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nLower 'suspicious long words' = fewer merged words.")

for name, res in s03_results.items():
    _, bad = suspicious_tokens("\n".join(r["clean"] for r in res["kept"]))
    print(f"\nExamples from {name}:", bad[:6])

 extractor  words  suspicious long words  per 1000 words  avg word length
   pymupdf   8942                     16             1.8             5.55
pdfplumber   7271                    125            17.2             6.83

Lower 'suspicious long words' = fewer merged words.

Examples from pymupdf: ['mapPartitionsWithIndex(func)', 'sample(withReplacement,', 'intersection(otherDataset)', 'distinct([numTasks]))', 'groupByKey([numTasks])', 'aggregateByKey(zeroValue)(seqOp,']

Examples from pdfplumber: ['Cloudcomputingisamodelforenablingconvenient,', 'computingresources(e.g.,networks,servers,storage,', 'applications,andservices)thatcanberapidlyprovisioned', 'andreleasedwithminimalmanagementeffortorservice', 'Thiscloudmodelpromotesavailability', 'andiscomposedoffiveessentialcharacteristics,three']


In [12]:
def side_by_side(title_left, text_left, title_right, text_right, width=58, max_lines=30):
    """Print two texts in two columns so differences in spacing are easy to see."""
    def wrap(text):
        lines = []
        for line in text.splitlines():
            lines.extend(textwrap.wrap(line, width) or [""])
        return lines[:max_lines]

    print(f"{title_left:<{width}} | {title_right}")
    print("-" * width + "-+-" + "-" * width)
    for left, right in zip_longest(wrap(text_left), wrap(text_right), fillvalue=""):
        print(f"{left:<{width}} | {right}")


def by_page(name):
    return {r["page"]: r for r in s03_results[name]["records"]}


plumber_pages, mupdf_pages = by_page("pdfplumber"), by_page("pymupdf")
common = sorted(set(plumber_pages) & set(mupdf_pages))

# Sample 1: a page from the middle of the document
middle_page = common[len(common) // 2]
# Sample 2: the page where pdfplumber has the most suspicious long words (worst case)
worst_page = max(common, key=lambda p: len(suspicious_tokens(plumber_pages[p]["clean"])[1]))

for label, page in [("middle page", middle_page), ("worst page for pdfplumber", worst_page)]:
    print("\n" + "=" * 125)
    print(f"S03 PAGE {page} ({label})")
    print("=" * 125)
    side_by_side("PYMUPDF", mupdf_pages[page]["clean"], "PDFPLUMBER", plumber_pages[page]["clean"])


S03 PAGE 70 (middle page)
PYMUPDF                                                    | PDFPLUMBER
-----------------------------------------------------------+-----------------------------------------------------------
Big Data Platforms (5 ECTS)                                | Implementing RDD Operations and Actions
37/43                                                      | Implementing RDD Operations and Actions
Implementing RDD Operations and Actions                    | 
Implementing RDD Operations and Actions                    | As the RDD operations are run in several computers, there
As the RDD operations are run in several computers, there  | is no shared memory to use to share state between
is no shared memory to use to share state between          | operations
operations                                                 | 
The functions run in all of the distributed computers need | The functions run in all of the distributed computers need
to copy all of the data they need

In [13]:
# Write BOTH candidate outputs. The final choice is made after reviewing the comparison above.
for name, res in s03_results.items():
    out_path = write_pdf_text("S03", res["kept"], suffix=f"_{name}")
    chars = sum(len(r["clean"]) for r in res["kept"])
    print(f"Written: {out_path.relative_to(settings.PROJECT_ROOT)}  ({chars:,} characters)")
    add_summary(f"S03 ({name}, DECISION PENDING)", "PDF", f"{len(res['kept'])} pages ({len(res['records'])} in range)",
                chars, str(out_path.relative_to(settings.PROJECT_ROOT)))
    manifest.append({"source_id": "S03", "source_type": s03["source_type"], "format": "pdf", "extractor": name,
                     "output": out_path.name, "pages": len(res["kept"]), "characters": chars,
                     "page_markers": True, "status": "candidate - extractor decision pending"})

Written: data\cleaned\S03_pymupdf.txt  (58,435 characters)
Written: data\cleaned\S03_pdfplumber.txt  (57,206 characters)


## 6. S06 - Data Engineering Zoomcamp (Markdown, SQL and Python in a ZIP)
Rules from `settings.py`:
- keep only the seven module folders `01-...` to `07-...`; skip `cohorts/`, `.tmp/`, `.github/`, `projects/`, `scripts/` and images;
- keep `.md`, `.sql` and `.py` files; `.md` is labelled `course_notes`, `.sql` and `.py` are labelled `code`;
- strip the front matter (`prev_url`, `next_url`, ...) from the Markdown files.

After extraction, the code files of `07-streaming` are measured to decide whether they are useful or noise.

In [14]:
s06 = settings.SOURCES["S06"]
archive6, entries6 = list_zip_files(settings.RAW_DIR / s06["path"])

include_ext = set(s06["include_extensions"])
exclude_ext = set(s06["exclude_extensions"])
skipped = {"not in a module folder": 0, "excluded folder": 0, "extension not wanted": 0}
selected6 = []
for info, rel in entries6:
    parts = rel.split("/")
    suffix = Path(rel).suffix.lower()
    if any(p in s06["exclude_folders"] for p in parts[:-1]):
        skipped["excluded folder"] += 1
    elif parts[0] not in s06["include_folders"]:
        skipped["not in a module folder"] += 1
    elif suffix not in include_ext or suffix in exclude_ext:
        skipped["extension not wanted"] += 1
    else:
        selected6.append((info, rel))

print(f"S06: {len(entries6)} files in the ZIP -> {len(selected6)} selected")
print("Skipped:", skipped)

S06: 954 files in the ZIP -> 185 selected
Skipped: {'not in a module folder': 20, 'excluded folder': 197, 'extension not wanted': 552}


In [15]:
out_dir6 = settings.CLEANED_DIR / "S06"
reset_output_folder(out_dir6)

s06_records = []
sample_shown = False
for info, rel in selected6:
    raw = archive6.read(info).decode("utf-8", errors="replace")
    suffix = Path(rel).suffix.lower()
    source_type = s06["source_type_by_extension"][suffix]
    if suffix == ".md":
        cleaned = clean_markdown(raw, s06)           # strip front matter, tidy whitespace
    else:
        cleaned = raw.replace("\r\n", "\n")          # code: keep exactly as written (indentation matters)
    target = out_dir6 / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(cleaned, encoding="utf-8")

    s06_records.append({"module": rel.split("/")[0], "file": rel, "extension": suffix,
                        "source_type": source_type, "characters": len(cleaned)})
    manifest.append({"source_id": "S06", "source_type": source_type, "format": suffix.lstrip("."),
                     "original_path": rel, "output": f"S06/{rel}", "characters": len(cleaned)})
    if not sample_shown and suffix == ".md" and raw.startswith("---"):
        show_before_after(f"S06 - {rel}", raw, cleaned, "front matter removed")
        sample_shown = True

s06_df = pd.DataFrame(s06_records)
s06_total = int(s06_df["characters"].sum())
print(f"S06: wrote {len(s06_df)} files, {s06_total:,} characters")
add_summary("S06", "Markdown + code (ZIP)", f"{len(s06_df)} files", s06_total, f"{len(s06_df)} files in data/cleaned/S06/")

S06 - 01-docker-terraform/01-introduction.md (front matter removed)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
---
next_url: 02-virtual-environment.md
---
# Introduction to Docker

Docker is a _containerization software_ that allows us to isolate software in a similar way to virtual machines but in a much leaner way.

A Docker image is a _snapshot_ of a container that we can define to run our software, or in this case our data pipelines. By exporting our Docker images to Cloud providers such as Amazon Web 
------------------------------------------------------------------------------------------
AFTER (cleaned text):
# Introduction to Docker

Docker is a _containerization software_ that allows us to isolate software in a similar way to virtual machines but in a much leaner way.

A Docker image is a _snapshot_ of a container that we can define to run our software, or in this case our data pipelines. By exportin

S06: wrote 185 files, 591,081 characters

### S06 - how much do the code files contribute?
Characters and files per module and type. "Estimated chunks" is only a rough guide (characters divided by the chunk size and overlap from `settings.py`); the real chunking happens in notebook 02.

In [16]:
def estimated_chunks(characters):
    """Rough chunk count for one file using settings.CHUNK_SIZE and settings.CHUNK_OVERLAP."""
    step = settings.CHUNK_SIZE - settings.CHUNK_OVERLAP
    return max(1, math.ceil(max(characters - settings.CHUNK_OVERLAP, 1) / step))


s06_df["est_chunks"] = s06_df["characters"].apply(estimated_chunks)

table = (s06_df.groupby(["module", "source_type", "extension"])
         .agg(files=("file", "count"), characters=("characters", "sum"), est_chunks=("est_chunks", "sum"))
         .reset_index())
print(table.to_string(index=False))

by_type = s06_df.groupby("source_type").agg(files=("file", "count"), characters=("characters", "sum"),
                                            est_chunks=("est_chunks", "sum"))
by_type["% of S06 characters"] = (100 * by_type["characters"] / s06_total).round(1)
print("\nS06 by source_type:")
print(by_type.to_string())

                   module  source_type extension  files  characters  est_chunks
      01-docker-terraform         code       .py      1        2373           4
      01-docker-terraform course_notes       .md     18       52150          78
02-workflow-orchestration course_notes       .md     19       34247          55
        03-data-warehouse         code       .py      3       14202          22
        03-data-warehouse         code      .sql      3        5427           9
        03-data-warehouse course_notes       .md      8       47452          71
 04-analytics-engineering         code      .sql     11       11535          21
 04-analytics-engineering course_notes       .md     16      107151         158
        05-data-platforms course_notes       .md     11       36023          56
                 06-batch         code       .py      2        5203           8
                 06-batch course_notes       .md     21      125513         187
             07-streaming         code  

In [17]:
streaming_py = s06_df[(s06_df["module"] == "07-streaming") & (s06_df["extension"] == ".py")]
streaming_all = s06_df[s06_df["module"] == "07-streaming"]
streaming_md = streaming_all[streaming_all["extension"] == ".md"]

print("07-streaming .py files")
print(f"  files                    : {len(streaming_py)}")
print(f"  characters               : {streaming_py['characters'].sum():,}")
print(f"  estimated chunks         : {streaming_py['est_chunks'].sum()}")
print(f"  share of the 07 module   : {100 * streaming_py['characters'].sum() / streaming_all['characters'].sum():.1f}% of its characters")
print(f"  share of all of S06      : {100 * streaming_py['characters'].sum() / s06_total:.1f}% of its characters")
print(f"  for comparison, 07 .md   : {len(streaming_md)} files, {streaming_md['characters'].sum():,} characters, "
      f"{streaming_md['est_chunks'].sum()} estimated chunks")

print("\nLargest 07-streaming .py files:")
print(streaming_py.sort_values("characters", ascending=False).head(8)[["file", "characters"]].to_string(index=False))

print("\nSample of the first 07-streaming .py file (to judge if it is useful explanation or boilerplate):")
sample_file = streaming_py.sort_values("characters", ascending=False).iloc[0]["file"]
print(f"--- {sample_file} ---")
print((out_dir6 / sample_file).read_text(encoding="utf-8")[:600])

07-streaming .py files
  files                    : 46
  characters               : 74,778
  estimated chunks         : 125
  share of the 07 module   : 49.9% of its characters
  share of all of S06      : 12.7% of its characters
  for comparison, 07 .md   : 26 files, 75,027 characters, 117 estimated chunks

Largest 07-streaming .py files:
                                                                   file  characters
 07-streaming/extras/extras/python/streams-example/pyspark/streaming.py        4243
07-streaming/extras/extras/python/streams-example/redpanda/streaming.py        4234
             07-streaming/extras/extras/python/avro_example/producer.py        3758
                 07-streaming/extras/extras/pyflink/src/job/taxi_job.py        3698
          07-streaming/extras/extras/pyflink/src/job/aggregation_job.py        3145
             07-streaming/extras/extras/python/avro_example/consumer.py        3110
                      07-streaming/code/src/job/aggregation_job_demo.p

## 7. Manifest and summary

In [18]:
manifest_path = settings.CLEANED_DIR / "extraction_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Manifest written: {manifest_path.relative_to(settings.PROJECT_ROOT)} ({len(manifest)} entries)\n")

summary = pd.DataFrame(summary_rows)
summary["characters extracted"] = summary["characters extracted"].map("{:,}".format)
summary = summary.rename(columns={"source": "source ID"})
print(summary.to_string(index=False))

Manifest written: data\cleaned\extraction_manifest.json (267 entries)

                         source ID                format    pages/files processed characters extracted          output file(s) written
                               S01                   PDF 142 pages (143 in range)              197,756            data\cleaned\S01.txt
                               S02                   PDF   94 pages (94 in range)              231,551            data\cleaned\S02.txt
                               S04                   PDF   70 pages (70 in range)              207,021            data\cleaned\S04.txt
                               S07                   PDF     9 pages (9 in range)               13,327            data\cleaned\S07.txt
                               S05        Markdown (ZIP)                 76 files              186,833   76 files in data/cleaned/S05/
   S03 (pymupdf, DECISION PENDING)                   PDF 138 pages (138 in range)               58,435    data\cleaned\

## 8. Checklist before moving on
- [ ] S03: choose PyMuPDF or pdfplumber (Section 5), then delete the unused `data/cleaned/S03_*.txt` and set the choice in `settings.py`.
- [ ] S06: decide whether to keep, trim or drop the `07-streaming` `.py` files (Section 6).
- [ ] S07: confirm the header line is gone in the AFTER sample; check whether page footers also need a rule.
- [ ] Spot-check one cleaned file per source in `data/cleaned/`.
- [ ] Then continue with `02_chunking.ipynb`.